# Spambase

First experiment using the shared workflow in `ml_workflow.py`.
The goal is to classify emails as spam (`1`) or non-spam (`0`).


In [1]:
import pandas as pd

from ml_workflow import (
    evaluate_classifier,
    load_spambase,
    split_dataset,
    train_logistic_baseline,
)


## Dataset

We check the number of samples, the number of features, missing values,
and the class distribution.


In [2]:
X, y = load_spambase()

print(f"Samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Missing feature values: {X.isna().sum().sum()}")


Samples: 4601
Features: 57
Missing feature values: 0


In [3]:
class_distribution = pd.DataFrame({
    "count": y.value_counts().sort_index(),
    "proportion": y.value_counts(normalize=True).sort_index(),
})
class_distribution.index.name = "spam"
class_distribution.round(3)


,count,proportion
spam,,
0,2788,0.606
1,1813,0.394


## Train/test split

We use 80% of the data for training and 20% for testing.
The split is stratified to preserve similar class proportions.
The random seed is fixed at 42.


In [4]:
X_train, X_test, y_train, y_test = split_dataset(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")


Training samples: 3680
Test samples: 921


## Logistic regression

The pipeline applies median imputation and standardization before
logistic regression. The preprocessing steps are fitted on the training split.


In [5]:
model = train_logistic_baseline(X_train, y_train)


## Test results

Precision, recall, and F1-score refer to the spam class (`1`).


In [6]:
scores = evaluate_classifier(model, X_test, y_test)
results = pd.Series(scores, name="Logistic regression").to_frame()
results.round(3)


,Logistic regression
accuracy,0.929
balanced_accuracy,0.924
precision,0.921
recall,0.898
f1,0.909
roc_auc,0.970
average_precision,0.951


On this split, logistic regression reaches about 92.9% accuracy and an F1-score of 0.909. Precision is 92.1% and recall is 89.8% for spam.

This gives a first reference for later model comparisons. Model and feature selection will use cross-validation on the training split.


## Cross-validation

We evaluate the pipeline using five stratified folds of the training split.
The table reports the mean and standard deviation of each metric.

In [7]:
from ml_workflow import cross_validate_classifier

cv_scores = cross_validate_classifier(model, X_train, y_train)
cv_scores.agg(["mean", "std"]).T.round(3)

,mean,std
accuracy,0.923,0.012
balanced_accuracy,0.915,0.012
precision,0.920,0.023
recall,0.881,0.019
f1,0.900,0.015
roc_auc,0.971,0.007
average_precision,0.947,0.015


## Class weighting

In [8]:
balanced_model = train_logistic_baseline(
    X_train, y_train, class_weight="balanced"
)

balanced_cv_scores = cross_validate_classifier(
    balanced_model, X_train, y_train, random_state=42
)

comparison = pd.DataFrame({
    "standard": cv_scores.mean(),
    "balanced": balanced_cv_scores.mean(),
}).T

comparison.round(3)

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
standard,0.923,0.915,0.920,0.881,0.900,0.971,0.947
balanced,0.928,0.925,0.903,0.914,0.909,0.971,0.947


Class weighting increases recall from 0.881 to 0.914, while precision decreases from 0.920 to 0.903. F1 improves slightly from 0.900 to 0.909. The weighted model detects more spam at the cost of additional false positives.

## Comparing classifiers

In [9]:
from ml_workflow import build_models, compare_models

SEED = 42  

models = build_models(random_state=SEED)

comparison, fold_scores = compare_models(
    models,
    X_train,
    y_train,
    random_state=SEED,
)

mean_scores = comparison.xs("mean", axis=1, level=1)
mean_scores.sort_values("f1", ascending=False).round(3)

Cross-validating Logistic regression
Cross-validating Logistic regression (balanced)
Cross-validating Gaussian Naive Bayes
Cross-validating Linear SVM (balanced)
Cross-validating MLP


,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
model,,,,,,,
Logistic regression (balanced),0.928,0.925,0.903,0.914,0.909,0.971,0.947
Linear SVM (balanced),0.924,0.923,0.895,0.916,0.905,0.970,0.947
Logistic regression,0.923,0.915,0.920,0.881,0.900,0.971,0.947
MLP,0.917,0.911,0.901,0.886,0.893,0.966,0.948
Gaussian Naive Bayes,0.814,0.838,0.693,0.951,0.802,0.917,0.826


Balanced logistic regression achieves the highest mean F1-score (0.909), followed closely by the balanced linear SVM (0.905). Gaussian Naive Bayes has the highest recall (0.951), but its precision of 0.693 leads to a lower F1-score of 0.802. Using F1 as our main selection criterion, we retain the two balanced linear models for the feature-selection experiment.

## Feature selection

In [10]:
from ml_workflow import search_feature_selection

candidate_names = [
    "Logistic regression (balanced)",
    "Linear SVM (balanced)",
]
candidates = {name: models[name] for name in candidate_names}

feature_results, feature_searches = search_feature_selection(
    candidates,
    X_train,
    y_train,
    random_state=SEED,
)

feature_results.round(3)

Searching feature selection and C for Logistic regression (balanced)
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Searching feature selection and C for Linear SVM (balanced)
Fitting 5 folds for each of 12 candidates, totalling 60 fits


,model,percentile,C,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,f1_std
0,Linear SVM (balanced),100,0.1,0.928,0.926,0.902,0.916,0.909,0.970,0.947,0.011
1,Logistic regression (balanced),100,1.0,0.928,0.925,0.903,0.914,0.909,0.971,0.947,0.011
2,Logistic regression (balanced),100,10.0,0.926,0.925,0.897,0.918,0.907,0.971,0.948,0.014
3,Linear SVM (balanced),100,1.0,0.924,0.923,0.895,0.916,0.905,0.970,0.947,0.012
4,Linear SVM (balanced),100,10.0,0.924,0.923,0.895,0.916,0.905,0.970,0.948,0.013
5,Logistic regression (balanced),75,10.0,0.924,0.921,0.902,0.907,0.904,0.969,0.946,0.013
6,Logistic regression (balanced),100,0.1,0.924,0.921,0.901,0.906,0.904,0.968,0.942,0.009
7,Logistic regression (balanced),75,1.0,0.924,0.920,0.904,0.903,0.903,0.969,0.945,0.011
8,Linear SVM (balanced),75,10.0,0.923,0.920,0.901,0.904,0.902,0.968,0.947,0.013
9,Linear SVM (balanced),75,1.0,0.923,0.919,0.901,0.903,0.902,0.969,0.946,0.013


### Best configuration for each classifier

We report the configuration with the highest mean cross-validation F1-score for each classifier.

In [11]:
feature_results.groupby("model", sort=False).head(1).round(3)

,model,percentile,C,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,f1_std
0,Linear SVM (balanced),100,0.1,0.928,0.926,0.902,0.916,0.909,0.970,0.947,0.011
1,Logistic regression (balanced),100,1.0,0.928,0.925,0.903,0.914,0.909,0.971,0.947,0.011


### Selected features

We display the best parameters and the feature names retained by each pipeline after refitting it on the full training split.

In [12]:
for name, search in feature_searches.items():
    features = search.best_estimator_[:-1].get_feature_names_out()
    print(name, search.best_params_)
    print(f"{len(features)} selected features:", features.tolist())

Logistic regression (balanced) {'classifier__C': 1.0, 'select__percentile': 100}
57 selected features: ['word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d', 'word_freq_our', 'word_freq_over', 'word_freq_remove', 'word_freq_internet', 'word_freq_order', 'word_freq_mail', 'word_freq_receive', 'word_freq_will', 'word_freq_people', 'word_freq_report', 'word_freq_addresses', 'word_freq_free', 'word_freq_business', 'word_freq_email', 'word_freq_you', 'word_freq_credit', 'word_freq_your', 'word_freq_font', 'word_freq_000', 'word_freq_money', 'word_freq_hp', 'word_freq_hpl', 'word_freq_george', 'word_freq_650', 'word_freq_lab', 'word_freq_labs', 'word_freq_telnet', 'word_freq_857', 'word_freq_data', 'word_freq_415', 'word_freq_85', 'word_freq_technology', 'word_freq_1999', 'word_freq_parts', 'word_freq_pm', 'word_freq_direct', 'word_freq_cs', 'word_freq_meeting', 'word_freq_original', 'word_freq_project', 'word_freq_re', 'word_freq_edu', 'word_freq_table', 'word_freq_confere

The highest mean CV F1-scores are obtained using all 57 features. The balanced linear SVM selects C=0.1, while balanced logistic regression selects C=1. Both achieve approximately 0.909 F1. Reducing the feature set lowers performance, particularly when retaining only 50% or 25% of the features. We therefore retain all features for the final evaluation.